In [27]:
pip install -U pydantic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Not uninstalling pydantic-core at /opt/conda/lib/python3.11/site-packages, outside environment /root/.clearml/venvs-builds/3.11
    Can't uninstall 'pydantic_core'. No files were found to uninstall.
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.22
    Uninstalling pydantic-1.10.22:
      Successfully uninstalled pydantic-1.10.22
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.89.1 requires pydantic!=1.7,!=1.7.1,!=1.7.2,!=1.7.3,!=1.8,!=1.8.1,<2.0.0,>=1.6.2, but you have pydantic 2.11.7 which is incompatible.
mistral-common 1.5.3 requires numpy>=1.25; python_version >= "3.9", but you have numpy 1.23.5 which is incompatible.
vllm 0.6.6 requires fastap

In [1]:
import json
import torch
import numpy as np

In [2]:
from deeppavlov import build_model, train_model, train_evaluate_model_from_config, evaluate_model
from deeppavlov.core.common.file import read_json

In [3]:
from deeppavlov.dataset_readers.hallucination_detection_reader import HallucinationDatasetReader, RAGTruthDatasetReader

In [4]:
from transformers import AutoModel, AutoConfig, AutoTokenizer, AutoModelForTokenClassification

In [5]:
from deeppavlov.core.data.data_learning_iterator import DataLearningIterator
from deeppavlov.models.preprocessors.torch_transformers_preprocessor import TorchTransformersHallucinationDetectorPreprocessor 
from deeppavlov.metrics.fmeasure import token_binary_f1, token_binary_precision, token_binary_recall

2025-06-30 16:03:37.904 WARNING in 'deeppavlov.core.common.registry'['registry'] at line 56: Registry name "torch_transformers_hallucination_detector_postprocessor" has been already registered and will be overwritten.


In [6]:
device = 'cuda'

In [7]:
path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_large.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_mbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_deberta_small.json'


# path = 'deeppavlov/configs/hallucination_detection/ragtruth_eurobert_base.json'

In [8]:
config = read_json(path)

In [9]:
model = build_model(config, load_trained=True)

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
contexts = ["France is a country in Europe. The capital of France is Paris. The population of France is 67 million.",]
question = "What is the capital of France? What is the population of France?"
answer = "The capital of France is Paris. The population of France is 69 million."

In [30]:
sample = {
    'context': contexts,
    'question': question,
    'answer': answer,
    "labels": [],
}

In [31]:
out = model([sample])

In [32]:
spans = out[-1][0]

In [33]:
spans

[{'start': 31,
  'end': 71,
  'confidence': 0.984317421913147,
  'text': ' The population of France is 69 million.'}]

In [36]:
url = "http://localhost:5000/model"
response = requests.post(url, json={"x": [sample]})
print(response.json()[-1])

[[{'start': 31, 'end': 71, 'confidence': 0.984317421913147, 'text': ' The population of France is 69 million.'}]]


In [35]:
print(response.json()[-1])

[[{'start': 31, 'end': 71, 'confidence': 0.984317421913147, 'text': ' The population of France is 69 million.'}]]


In [ ]:
requests.

In [32]:
probabilities = out[1]

In [ ]:
_, _, offsets, answer_start_token = TorchTransformersHallucinationDetectorPreprocessor.prepare_tokenized_input(
    self.tokenizer, sample['prompt'], sample['answer'], self.max_seq_length
)

In [17]:
out[0]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0]])